# Pipeline
0. Get the list of required swc files
1. Load labels parquet
2. Import the swc file
3. simplify swc file
4. attach synapse labels + neuron type
5. save simplified file
6. convert to json for find-clumpiness
7. save json file
8. calculate clumpiness for each internal node
9. attach results to the labeled swc file
10. save results.

---
# Preprocessing step
1. Create metadata labels for each swc file
2. Look for the releveant swc files only (with the wanted type)
3. Unify them via the already created function in feather file ->>> Improvement

In [1]:
import os
import json
import numpy as np
import pandas as pd
import polars as pl
from tqdm import tqdm
from scripts.helpers import mkdir
from joblib import Parallel, delayed
from scripts.preprocessing import simplify_swc_topology, swc2json, get_neurons_info
from scripts.processing import generate_internal_subtrees

if False:
    # Type data located in the 
    path_swc_labels = os.path.join("data", "input_labels", "neuron_data_full_article_princeton.ftr")
    swc_labels = pd.read_feather(path_swc_labels)

    required_labels = ["super_class", ["central", "optic", "visual_centrifugal", "visual_projection"]]
    swc_labels = swc_labels.loc[swc_labels[required_labels[0]].isin(required_labels[1])]

-----
# Single-file hard coded pipeline example

In [2]:
if False:
    nueron_itr = 720575940609102805


    ####################################################################################################
    #  1. Load labels parquet
    # Parquet labels path
    prquet_labels_path = os.path.join("data", "input_labels", "swc_labels.parquet")

    # Load exactly the labels of the example swc file
    parquet_labels = pl.scan_parquet(prquet_labels_path)

    # Only the relevnt column in the parquet file
    labels_parquet = parquet_labels.select(["neuron", "node_id", "type"]).filter(pl.col("neuron") == str(nueron_itr)).collect().to_pandas()


    ####################################################################################################
    #  2. Import the swc file
    neuron_path = os.path.join("data","input_swc", "sk_lod1_783_healed", f"{nueron_itr}.swc")
    neuron_swc = pd.read_csv(neuron_path, 
                            comment='#', 
                            header=None, 
                            sep=r'\s+', 
                            names=["node_id", "swc_type", "x", "y", "z", "r", "parent"])


    ####################################################################################################
    #  3. simplify swc file
    simple_swc = simplify_swc_topology(neuron_swc, swc_name=f"{nueron_itr}", save_csv=False)


    ####################################################################################################
    #  4. attach synapse labels + neuron type
    swc_labeled = pd.merge(left=simple_swc, 
                        right=labels_parquet[["node_id", "type"]].drop_duplicates(), 
                        left_on="node_id", 
                        right_on="node_id", 
                        how="left")


    ####################################################################################################
    #  5. save simplified file
    for i in ["data", os.path.join("data", "input_swc"), os.path.join("data", "input_swc", "simplified")]:
        if os.path.exists(i) is False:
            os.mkdir(i)

    save_path = os.path.join("data", "input_swc", "simplified", f"{nueron_itr}.csv")
    swc_labeled.to_csv(save_path)


    ####################################################################################################
    #  6. convert to json for find-clumpiness
    swc2json(swc_dataset=swc_labeled,
            neuron_id=nueron_itr,
            save_json=True,
            save_path=os.path.join("data", "output_json"))


----
# Automated script pipeline


In [ ]:
def process_neuron(neuron_itr):
    # Force Polars to use a single thread to prevent nested parallelism crashes
    os.environ["POLARS_MAX_THREADS"] = "4"

    try:
        #########################
        #  1. Load labels parquet
        # Parquet labels path
        prquet_labels_path = os.path.join("data", "input_labels", "swc_labels.parquet")


        # Load exactly the labels of the example swc file
        parquet_labels = pl.scan_parquet(prquet_labels_path)


        # Only the relevnt column in the parquet file
        labels_parquet = parquet_labels.select(["neuron", "node_id", "type"]).filter(pl.col("neuron") == str(neuron_itr)).collect().to_pandas()


        #########################
        #  2. Import the swc file
        neuron_path = os.path.join("data","input_swc", "sk_lod1_783_healed", f"{neuron_itr}.swc")
        neuron_swc = pd.read_csv(neuron_path, 
                                 comment='#', 
                                 header=None, 
                                 sep=r'\s+', 
                                 names=["node_id", "swc_type", "x", "y", "z", "r", "parent"])


        #######################
        #  3. simplify swc file
        simple_swc = simplify_swc_topology(neuron_swc, swc_name=f"{neuron_itr}", save_csv=False)  


        #########################################
        #  4. attach synapse labels + neuron type
        swc_labeled = pd.merge(left=simple_swc, 
                               right=labels_parquet[["node_id", "type"]].drop_duplicates(), 
                               left_on="node_id", 
                               right_on="node_id", 
                               how="left")   


        ##########################
        #  5. save simplified file
        for i in ["data", os.path.join("data", "input_swc"), os.path.join("data", "input_swc", "simplified")]:
            if os.path.exists(i) is False:
                os.mkdir(i)

        save_path = os.path.join("data", "input_swc", "simplified", f"{neuron_itr}.csv")
        swc_labeled["type"] = swc_labeled.groupby("node_id")["type"].unique().apply(lambda X : X[0] if len(X) <= 1 else ",".join(X)) # joining labels if more then 2 per node
        swc_labeled = swc_labeled.drop_duplicates(subset=["node_id", "parent"], keep="first")                                        # dropping rows with the same parent+node_id
        swc_labeled.to_csv(save_path)


        #########################################
        #  6. convert to json for find-clumpiness
        swc2json(swc_dataset=swc_labeled.drop_duplicates(),
                 neuron_id=neuron_itr,
                 save_json=True,
                 save_path=os.path.join("data", "output_json"))


        ##################################################################
        # 7. Devide main tree to multiple sub-trees for each internal node
        # Example Usage:
        generate_internal_subtrees(input_json_path = os.path.join("data","output_json",f"{neuron_itr}_0.json"), 
                                   neuron_number = neuron_itr, 
                                   output_dir = os.path.join("data", "output_json"))
        

        return f"Success: {neuron_itr}"

    except Exception as e:
        return f"Error on {neuron_itr}: {e}"


if __name__ == '__main__':
    # Define paths
    swc_path = os.path.join("data", "input_swc", "sk_lod1_783_healed")
    labels_path = os.path.join("data", "input_labels", "processed_swc_data_princeton")
    prquet_labels_path = os.path.join("data", "input_labels", "swc_labels.parquet")

    # Safe folder creation before multiprocessing starts to avoid race conditions
    folders_to_create = ["data", 
                         os.path.join("data", "input_swc"), 
                         os.path.join("data", "input_swc", "simplified"),
                         os.path.join("data", "output_json")]
    
    for folder in folders_to_create:
        os.makedirs(folder, exist_ok=True)

    # Getting a list of the aviable SWC file in the swc input folder
    swc_files = [i.split(".")[0] for i in os.listdir(swc_path)]

    # Creating parquete file -> only relevent swc file by super-type
    get_neurons_info(overwrite_parquet=False)

    # Getting relevent swc
    parquet_labels = pl.scan_parquet(prquet_labels_path)
    labels_parquet = parquet_labels.select(["neuron"]).collect().to_pandas()

    # Getting list of relevent + real SWC file
    swc_relv = np.intersect1d(swc_files, labels_parquet.neuron.values)
    
    # Select the batch you want to run
    tasks = swc_relv[:20] 
    
    print(f"Starting processing of {len(tasks)} neurons...")
    
    # Execute in parallel using Joblib
    # n_jobs=4 limits the pool to 4 cores to prevent memory exhaustion. 
    # You can increase this if your system has plenty of RAM.
    results = Parallel(n_jobs=4, backend="loky")(delayed(process_neuron)(neuron) for neuron in tqdm(tasks))
    
    # Print any errors that were caught during execution
    for res in results:
        if "Error" in res:
            print(res)


> Function execution halted, old `swc_labels.parquet` file preserved.
Starting processing of 20 neurons...


100%|██████████| 20/20 [00:16<00:00,  1.22it/s]
